# Y-Offset Comparison Analysis

Compares WT simulations run with a y-offset of 88 (standard) vs a y-offset of 50
across three division-plane mean conditions (divMean=0°, 45°, 90°).

**Primary question:** does changing the y-offset of the initial NB position affect
spatial mixing (`norm_het_frac`) or NB boundary exposure independently of the
division-plane angle distribution?

**Secondary question:** does the y-offset alter lineage composition metrics
(lineage area, progeny counts, NB area)?

---
## Section 0 — Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [ ]:
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('Could not find repo root')
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / 'src'))
from npa.sim_viz import render_raw, load_raw_snapshot_full

SWEEP_ROOT  = REPO_ROOT / 'data/sim/sweep'
PROC_DIR    = REPO_ROOT / 'data/sim/processed_sweep'
FIGURES_DIR = PROC_DIR / 'figures/yoffset_comparison'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sim     = pd.read_csv(PROC_DIR / 'sim_metrics_last.csv')
run_idx = pd.read_csv(PROC_DIR / 'sim_run_index.csv', dtype=str)
run_idx['npz_row'] = run_idx['npz_row'].astype(int)

print('sim_metrics_last:', len(sim), 'rows')
print('Conditions found:', sorted(sim['condition'].unique()))

In [ ]:
import re as _re

# ── Sim-series metadata ───────────────────────────────────────────────────
SIM_ID_META = {
    'sim61': {'regulatory_dynamic': 'NONE',    'critical_volume_mode': 1, 'genotype': 'wt'},
    'sim62': {'regulatory_dynamic': 'NB-ABM',  'critical_volume_mode': 1, 'genotype': 'wt'},
    'sim63': {'regulatory_dynamic': 'VOL-ABM', 'critical_volume_mode': 1, 'genotype': 'wt'},
    'sim64': {'regulatory_dynamic': 'NB-PDE',  'critical_volume_mode': 1, 'genotype': 'wt'},
    'sim65': {'regulatory_dynamic': 'VOL-PDE', 'critical_volume_mode': 1, 'genotype': 'wt'},
    'sim71': {'regulatory_dynamic': 'NONE',    'critical_volume_mode': 0, 'genotype': 'wt'},
    'sim72': {'regulatory_dynamic': 'NB-ABM',  'critical_volume_mode': 0, 'genotype': 'wt'},
    'sim73': {'regulatory_dynamic': 'VOL-ABM', 'critical_volume_mode': 0, 'genotype': 'wt'},
    'sim74': {'regulatory_dynamic': 'NB-PDE',  'critical_volume_mode': 0, 'genotype': 'wt'},
    'sim75': {'regulatory_dynamic': 'VOL-PDE', 'critical_volume_mode': 0, 'genotype': 'wt'},
}

_meta_df = pd.DataFrame(SIM_ID_META).T.reset_index().rename(columns={'index': 'sim_id'})
_meta_df['critical_volume_mode'] = _meta_df['critical_volume_mode'].astype(int)
sim = sim[sim['condition'].str.startswith('wt_')].copy()
sim = sim.drop(columns=[c for c in ['regulatory_dynamic', 'critical_volume_mode', 'genotype']
                         if c in sim.columns], errors='ignore')
sim = sim.merge(_meta_df, on='sim_id', how='left')

# ── Condition pairs: yoffset88 (standard) vs yoffset50 (new) ─────────────
PAIRS = [
    {'tag': 'D0S26',  'yoffset88': 'wt_divMean0Stdev26',  'yoffset50': 'wt_divMean0Stdev26_yoffset50'},
    {'tag': 'D45S26', 'yoffset88': 'wt_divMean45Stdev26', 'yoffset50': 'wt_divMean45Stdev26_yoffset50'},
    {'tag': 'D90S26', 'yoffset88': 'wt_divMean90Stdev26', 'yoffset50': 'wt_divMean90Stdev26_yoffset50'},
]
TAGS    = [p['tag'] for p in PAIRS]
OFFSETS = ['yoffset88', 'yoffset50']

COND_INFO = {}
for p in PAIRS:
    COND_INFO[p['yoffset88']] = {'tag': p['tag'], 'offset': 'yoffset88'}
    COND_INFO[p['yoffset50']] = {'tag': p['tag'], 'offset': 'yoffset50'}

sim['tag']    = sim['condition'].map(lambda c: COND_INFO.get(c, {}).get('tag', c))
sim['offset'] = sim['condition'].map(lambda c: COND_INFO.get(c, {}).get('offset', 'unknown'))
sim = sim[sim['offset'].isin(OFFSETS)].copy()

METRICS = ['lin_area_vox', 'n_pros', 'n_dpn', 'dpn_area_vox', 'avg_dpn_area_vox']
METRIC_LABELS = {
    'lin_area_vox':     'Lineage area (vox)',
    'n_pros':           'Pros count',
    'n_dpn':            'NB count',
    'dpn_area_vox':     'Total NB area (vox)',
    'avg_dpn_area_vox': 'Avg NB area (vox/cell)',
}

# Verify all expected conditions are present
present = set(sim['condition'].unique())
for p in PAIRS:
    for key in ['yoffset88', 'yoffset50']:
        flag = '\u2713' if p[key] in present else '\u2717 MISSING'
        print(f"{flag}  {p[key]}")

In [ ]:
import npa.sim_viz as _sim_viz_mod

_sim_viz_mod.POP2_COLOR = '#4a96b7'
_sim_viz_mod.POP3_COLOR = '#00a6a4'

NB_COLOR   = _sim_viz_mod.NB_COLOR
POP2_COLOR = _sim_viz_mod.POP2_COLOR
POP3_COLOR = _sim_viz_mod.POP3_COLOR

OFFSET_COLORS = {'yoffset88': '#5b7fcc', 'yoffset50': '#cc7b4a'}
REG_DYN_ORDER = ['NONE', 'NB-ABM', 'VOL-ABM', 'NB-PDE', 'VOL-PDE']

# ── Physical scale ────────────────────────────────────────────────────────
# pixels_per_um = canvas_size / sim_space_um
# yoffset50 sims: 400×400 µm space on 200×200 px canvas → 0.5 px/µm
# yoffset88 (standard) sims use _DEFAULT_PIXELS_PER_UM.
# Set _DEFAULT_PIXELS_PER_UM to canvas_size / sim_space_um for baseline sims.
PIXELS_PER_UM = {
    'wt_divMean0Stdev26_yoffset50':  200 / 400,
    'wt_divMean45Stdev26_yoffset50': 200 / 400,
    'wt_divMean90Stdev26_yoffset50': 200 / 400,
}
_DEFAULT_PIXELS_PER_UM = 1.0   # ← set to canvas_size / sim_space_um for baseline sims
SCALE_BAR_UM = 20              # ← width of scale bar label; adjust to taste
# ─────────────────────────────────────────────────────────────────────────


def heterotypic_contact_fraction(geo):
    nb    = geo[..., 0] > 0
    nonnb = geo[..., 1] > 0
    occ   = nb | nonnb
    n_occ = int(occ.sum())
    if n_occ == 0:
        return dict(het_frac=np.nan, norm_het_frac=np.nan,
                    p_nb=np.nan, p_nonnb=np.nan, n_occupied=0)
    p_nb_val    = float(nb[occ].mean())
    p_nonnb_val = float(nonnb[occ].mean())
    both_h = occ[:, :-1] & occ[:, 1:]
    het_h  = both_h & (nb[:, :-1] ^ nb[:, 1:])
    both_v = occ[:-1, :] & occ[1:, :]
    het_v  = both_v & (nb[:-1, :] ^ nb[1:, :])
    total = int(both_h.sum()) + int(both_v.sum())
    het   = int(het_h.sum())  + int(het_v.sum())
    if total == 0:
        return dict(het_frac=np.nan, norm_het_frac=np.nan,
                    p_nb=p_nb_val, p_nonnb=p_nonnb_val, n_occupied=n_occ)
    hf       = het / total
    expected = 2.0 * p_nb_val * p_nonnb_val
    norm_hf  = (hf / expected) if expected > 0 else np.nan
    return dict(het_frac=hf, norm_het_frac=norm_hf,
                p_nb=p_nb_val, p_nonnb=p_nonnb_val, n_occupied=n_occ)


def nb_exposure_fraction(geo):
    nb    = geo[..., 0] > 0
    nonnb = geo[..., 1] > 0
    emp   = ~(nb | nonnb)
    n_exposed = 0
    n_contact = 0
    for nb_mask, other_mask in [
        (nb[:, :-1] & ~nb[:, 1:],  (emp[:, 1:],  nonnb[:, 1:])),
        (nb[:, 1:]  & ~nb[:, :-1], (emp[:, :-1], nonnb[:, :-1])),
        (nb[:-1, :] & ~nb[1:, :],  (emp[1:, :],  nonnb[1:, :])),
        (nb[1:, :]  & ~nb[:-1, :], (emp[:-1, :], nonnb[:-1, :])),
    ]:
        e_mask, c_mask = other_mask
        n_exposed += int((nb_mask & e_mask).sum())
        n_contact += int((nb_mask & c_mask).sum())
    n_perimeter = n_exposed + n_contact
    if n_perimeter == 0:
        return dict(exposed_frac=np.nan, n_exposed=0, n_contact=0, n_perimeter=0)
    return dict(exposed_frac=n_exposed / n_perimeter,
                n_exposed=n_exposed, n_contact=n_contact, n_perimeter=n_perimeter)


def percentile_run_id(condition, sim_id, metric, pct, df=None):
    if df is None:
        df = sim
    sub = df[(df['condition'] == condition) & (df['sim_id'] == sim_id)].dropna(subset=[metric])
    if sub.empty:
        return None
    threshold = np.percentile(sub[metric], pct)
    idx = (sub[metric] - threshold).abs().idxmin()
    return str(sub.loc[idx, 'run_id']).zfill(4)


def draw_lineage(condition, sim_id, run_id, title='', ax=None):
    run_id = str(run_id).zfill(4)
    match = run_idx[
        (run_idx['condition'] == condition) &
        (run_idx['sim_id']    == sim_id) &
        (run_idx['run_id']    == run_id)
    ]
    if match.empty:
        raise ValueError(f'run_index missing: {condition} {sim_id} {run_id}')
    r = match.iloc[0]
    geo_raw, lmap = load_raw_snapshot_full(
        REPO_ROOT / r['cells_path'],
        REPO_ROOT / r['locs_path'],
    )
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    _sim_viz_mod.render_raw(geo_raw, label_map=lmap, ax=ax, title=title)
    ax.axis('off')
    ppu = PIXELS_PER_UM.get(condition, _DEFAULT_PIXELS_PER_UM)
    _sim_viz_mod.add_scale_bar(ax, pixels_per_um=ppu, length_um=SCALE_BAR_UM)
    return ax

---
## Section 1 — Heterotypic Contact Fraction

Computes `norm_het_frac` for every WT run in both y-offset conditions.

**Reference:** y=1 is random mixing. Values < 1 indicate NB and non-NB cells
are spatially segregated relative to their proportions.

In [ ]:
_HET_CSV = PROC_DIR / 'het_yoffset_metrics.csv'

if _HET_CSV.exists():
    het_df = pd.read_csv(_HET_CSV)
    print(f'Loaded cached het_yoffset_metrics.csv ({len(het_df)} rows)')
else:
    all_conds = {p[k] for p in PAIRS for k in ['yoffset88', 'yoffset50']}
    het_records = []
    for npz_path_str, group in run_idx[run_idx['condition'].isin(all_conds)].groupby('npz_path'):
        with np.load(REPO_ROOT / npz_path_str) as _d:
            _geo_all = _d['geo']
        for _, row in group.iterrows():
            result = heterotypic_contact_fraction(_geo_all[int(row['npz_row'])])
            het_records.append({
                'condition': row['condition'],
                'sim_id':    row['sim_id'],
                'run_id':    row['run_id'],
                **result,
            })
    het_df = pd.DataFrame(het_records)
    _meta = sim[['condition', 'sim_id', 'regulatory_dynamic',
                  'critical_volume_mode', 'tag', 'offset']].drop_duplicates()
    het_df = het_df.merge(_meta, on=['condition', 'sim_id'], how='left')
    het_df.to_csv(_HET_CSV, index=False)
    print(f'Computed and saved ({len(het_df)} rows)')

print()
print(het_df.groupby(['tag', 'offset'])['norm_het_frac']
      .agg(['mean', 'std', 'median']).round(4).to_string())

In [ ]:
# norm_het_frac: yoffset88 vs yoffset50, faceted by VCV mode
box_w   = 0.14
offsets = np.array([-0.10, 0.10])   # yoffset88 left, yoffset50 right
n_tags  = len(TAGS)

vcv_modes = sorted(het_df['critical_volume_mode'].dropna().unique().astype(int))
n_vcv = len(vcv_modes)

fig, axes = plt.subplots(n_vcv, 1, figsize=(max(6, n_tags * 2.5), 4 * n_vcv),
                          sharey=True, squeeze=False)

for ri, vcv in enumerate(vcv_modes):
    ax = axes[ri, 0]
    ax.axhline(1.0, color='gray', linestyle='--', linewidth=1.2, zorder=1)
    sub = het_df[het_df['critical_volume_mode'] == vcv].dropna(subset=['norm_het_frac'])

    for ti, tag in enumerate(TAGS):
        for ai, off in enumerate(OFFSETS):
            runs = sub[(sub['tag'] == tag) & (sub['offset'] == off)]['norm_het_frac']
            if runs.empty:
                continue
            x  = ti + offsets[ai]
            bp = ax.boxplot(
                [runs], positions=[x], widths=box_w,
                patch_artist=True,
                medianprops={'color': 'black', 'linewidth': 1.2},
                whiskerprops={'linewidth': 0.7},
                capprops={'linewidth': 0.7},
                flierprops={'marker': 'o', 'markersize': 1.5, 'linestyle': 'none'},
                manage_ticks=False, zorder=2,
            )
            bp['boxes'][0].set_facecolor(OFFSET_COLORS[off])
            bp['boxes'][0].set_alpha(0.8)
            med       = float(np.median(runs))
            upper_cap = bp['caps'][1].get_ydata()[0]
            ax.text(x, upper_cap + 0.0005, f'{med:.4f}',
                    ha='center', va='bottom', fontsize=6, rotation=90)

    ax.set_xlim(-0.5, n_tags - 0.5)
    ax.set_xticks(list(range(n_tags)))
    ax.set_xticklabels(TAGS, fontsize=9)
    ax.set_ylabel(f'VCV={vcv}\nnorm_het_frac', fontsize=8)
    ax.grid(axis='y', alpha=0.3)

legend_handles = [
    Line2D([0], [0], color='gray', linestyle='--', label='random mixing (y=1)'),
    Patch(facecolor=OFFSET_COLORS['yoffset88'], alpha=0.8, label='yoffset=88 (standard)'),
    Patch(facecolor=OFFSET_COLORS['yoffset50'], alpha=0.8, label='yoffset=50'),
]
fig.suptitle('norm_het_frac: yoffset88 vs yoffset50 — WT', fontsize=11)
fig.legend(handles=legend_handles, fontsize=8, loc='lower center',
           ncol=3, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=[0, 0.07, 1, 0.97])
fig.savefig(FIGURES_DIR / 'het_by_yoffset.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Section 1b — NB Perimeter Exposure

Fraction of the NB region's 4-connected perimeter facing extracellular space
vs. any other cell type. Does the initial NB y-position change how exposed
the NB remains at the final timepoint?

In [ ]:
_EXP_CSV = PROC_DIR / 'nb_exposure_yoffset_metrics.csv'

if _EXP_CSV.exists():
    nbexp_df = pd.read_csv(_EXP_CSV)
    print(f'Loaded cached nb_exposure_yoffset_metrics.csv ({len(nbexp_df)} rows)')
else:
    all_conds = {p[k] for p in PAIRS for k in ['yoffset88', 'yoffset50']}
    exp_records = []
    for npz_path_str, group in run_idx[run_idx['condition'].isin(all_conds)].groupby('npz_path'):
        with np.load(REPO_ROOT / npz_path_str) as _d:
            _geo_all = _d['geo']
        for _, row in group.iterrows():
            result = nb_exposure_fraction(_geo_all[int(row['npz_row'])])
            exp_records.append({
                'condition': row['condition'],
                'sim_id':    row['sim_id'],
                'run_id':    row['run_id'],
                **result,
            })
    nbexp_df = pd.DataFrame(exp_records)
    _meta = sim[['condition', 'sim_id', 'regulatory_dynamic',
                  'critical_volume_mode', 'tag', 'offset']].drop_duplicates()
    nbexp_df = nbexp_df.merge(_meta, on=['condition', 'sim_id'], how='left')
    nbexp_df.to_csv(_EXP_CSV, index=False)
    print(f'Computed and saved ({len(nbexp_df)} rows)')

print()
print(nbexp_df.groupby(['tag', 'offset'])['exposed_frac']
      .agg(['mean', 'std', 'median']).round(4).to_string())

In [ ]:
# NB exposure fraction: yoffset88 vs yoffset50, faceted by VCV mode
box_w   = 0.14
offsets = np.array([-0.10, 0.10])

vcv_modes = sorted(nbexp_df['critical_volume_mode'].dropna().unique().astype(int))
n_vcv = len(vcv_modes)

fig, axes = plt.subplots(n_vcv, 1, figsize=(max(6, n_tags * 2.5), 4 * n_vcv),
                          sharey=True, squeeze=False)

for ri, vcv in enumerate(vcv_modes):
    ax = axes[ri, 0]
    sub = nbexp_df[nbexp_df['critical_volume_mode'] == vcv].dropna(subset=['exposed_frac'])

    for ti, tag in enumerate(TAGS):
        for ai, off in enumerate(OFFSETS):
            runs = sub[(sub['tag'] == tag) & (sub['offset'] == off)]['exposed_frac']
            if runs.empty:
                continue
            x  = ti + offsets[ai]
            bp = ax.boxplot(
                [runs], positions=[x], widths=box_w,
                patch_artist=True,
                medianprops={'color': 'black', 'linewidth': 1.2},
                whiskerprops={'linewidth': 0.7},
                capprops={'linewidth': 0.7},
                flierprops={'marker': 'o', 'markersize': 1.5, 'linestyle': 'none'},
                manage_ticks=False, zorder=2,
            )
            bp['boxes'][0].set_facecolor(OFFSET_COLORS[off])
            bp['boxes'][0].set_alpha(0.8)
            med       = float(np.median(runs))
            upper_cap = bp['caps'][1].get_ydata()[0]
            ax.text(x, upper_cap + 0.005, f'{med:.3f}',
                    ha='center', va='bottom', fontsize=6, rotation=90)

    ax.set_xlim(-0.5, n_tags - 0.5)
    ax.set_xticks(list(range(n_tags)))
    ax.set_xticklabels(TAGS, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_ylabel(f'VCV={vcv}\nNB exposed fraction', fontsize=8)
    ax.grid(axis='y', alpha=0.3)

legend_handles = [
    Patch(facecolor=OFFSET_COLORS['yoffset88'], alpha=0.8, label='yoffset=88 (standard)'),
    Patch(facecolor=OFFSET_COLORS['yoffset50'], alpha=0.8, label='yoffset=50'),
]
fig.suptitle('NB perimeter fraction exposed to extracellular space — WT', fontsize=11)
fig.legend(handles=legend_handles, fontsize=8, loc='lower center',
           ncol=2, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=[0, 0.07, 1, 0.97])
fig.savefig(FIGURES_DIR / 'nb_exposure_by_yoffset.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Section 2 — Composition Metrics

Checks whether changing the y-offset alters lineage size or cell-type composition.
Shown as raw distributions (not log FC) so both conditions can be read on the same scale.

In [ ]:
# Composition metrics: direct box distributions, faceted by VCV mode
box_w   = 0.14
offsets = np.array([-0.10, 0.10])

vcv_modes = sorted(sim['critical_volume_mode'].dropna().unique().astype(int))
n_vcv = len(vcv_modes)

for metric in METRICS:
    fig, axes = plt.subplots(n_vcv, 1, figsize=(max(6, n_tags * 2.5), 4 * n_vcv),
                              sharey=True, squeeze=False)

    for ri, vcv in enumerate(vcv_modes):
        ax = axes[ri, 0]
        sub = sim[sim['critical_volume_mode'] == vcv].dropna(subset=[metric])

        for ti, tag in enumerate(TAGS):
            for ai, off in enumerate(OFFSETS):
                runs = sub[(sub['tag'] == tag) & (sub['offset'] == off)][metric].dropna()
                if runs.empty:
                    continue
                x  = ti + offsets[ai]
                bp = ax.boxplot(
                    [runs], positions=[x], widths=box_w,
                    patch_artist=True,
                    medianprops={'color': 'black', 'linewidth': 1.2},
                    whiskerprops={'linewidth': 0.7},
                    capprops={'linewidth': 0.7},
                    flierprops={'marker': 'o', 'markersize': 1.5, 'linestyle': 'none'},
                    manage_ticks=False, zorder=2,
                )
                bp['boxes'][0].set_facecolor(OFFSET_COLORS[off])
                bp['boxes'][0].set_alpha(0.8)
                med       = float(np.median(runs))
                upper_cap = bp['caps'][1].get_ydata()[0]
                ax.text(x, upper_cap * 1.01, f'{med:.0f}',
                        ha='center', va='bottom', fontsize=6, rotation=90)

        ax.set_xlim(-0.5, n_tags - 0.5)
        ax.set_xticks(list(range(n_tags)))
        ax.set_xticklabels(TAGS, fontsize=9)
        ax.set_ylabel(f'VCV={vcv}\n{METRIC_LABELS[metric]}', fontsize=8)
        ax.grid(axis='y', alpha=0.3)

    legend_handles = [
        Patch(facecolor=OFFSET_COLORS['yoffset88'], alpha=0.8, label='yoffset=88 (standard)'),
        Patch(facecolor=OFFSET_COLORS['yoffset50'], alpha=0.8, label='yoffset=50'),
    ]
    fig.suptitle(f'{METRIC_LABELS[metric]} — yoffset88 vs yoffset50 (WT)', fontsize=11)
    fig.legend(handles=legend_handles, fontsize=8, loc='lower center',
               ncol=2, bbox_to_anchor=(0.5, 0.0))
    fig.tight_layout(rect=[0, 0.07, 1, 0.97])
    fig.savefig(FIGURES_DIR / f'composition_{metric}_by_yoffset.png', dpi=120, bbox_inches='tight')
    plt.show()

---
## Section 3 — Example Lineages

Visualises the 10th, 50th, and 90th percentile lineages (by `lin_area_vox`) for
each divMean condition under both y-offsets. sim63 (VOL-ABM, VCV=1) is used as
the representative regulatory dynamic.

Colour key: **purple** = NB, **blue** = GMC, **teal** = neuron.

In [ ]:
# Example lineages: 10th / 50th / 90th percentile of lin_area_vox
# One figure per divMean, 2 rows (yoffset88 / yoffset50) × 3 cols (10th / 50th / 90th)

VIZ_SIM_ID  = 'sim63'    # VOL-ABM, VCV=1 — representative regulatory dynamic
PERCENTILES = [10, 50, 90]
SORT_METRIC = 'lin_area_vox'

for p in PAIRS:
    tag = p['tag']
    fig, axes = plt.subplots(
        2, 3, figsize=(12, 8),
        gridspec_kw={'hspace': 0.05, 'wspace': 0.05},
    )

    for ri, off in enumerate(OFFSETS):
        cond = p[off]
        for ci, pct in enumerate(PERCENTILES):
            ax = axes[ri, ci]
            run_id = percentile_run_id(cond, VIZ_SIM_ID, SORT_METRIC, pct)
            if run_id is None:
                ax.set_visible(False)
                continue
            row_title = f'{off}  |  {pct}th pct\nrun {run_id}'
            draw_lineage(cond, VIZ_SIM_ID, run_id, title=row_title, ax=ax)

    # Row labels
    for ri, off in enumerate(OFFSETS):
        axes[ri, 0].set_ylabel(off, fontsize=9, labelpad=4)

    # Column labels
    for ci, pct in enumerate(PERCENTILES):
        axes[0, ci].set_title(f'{pct}th percentile ({SORT_METRIC})', fontsize=8)

    fig.suptitle(f'{tag}  —  {VIZ_SIM_ID}  (yoffset88 vs yoffset50)', fontsize=11)
    fig.savefig(FIGURES_DIR / f'lineages_{tag}_{VIZ_SIM_ID}.png', dpi=120, bbox_inches='tight')
    plt.show()

---
## Section 4 — Interpretation

*(Analyst notes)*

**norm_het_frac (Section 1):**
- If yoffset50 ≈ yoffset88 → initial y-position does not affect spatial mixing at
  the final timepoint.
- If yoffset50 differs substantially → boundary effects or asymmetric growth from
  a shifted starting position alter how NB and progeny intermix.

**NB exposure (Section 1b):**
- A higher `exposed_frac` under yoffset50 would suggest the NB is positioned closer
  to an open boundary and retains more exposed perimeter by the final timepoint.

**Composition metrics (Section 2):**
- If `lin_area_vox` or progeny counts differ → the y-offset affects growth dynamics
  (e.g., boundary confinement limits lineage expansion). This would confound direct
  comparisons of spatial metrics without size-matching.
- If composition is stable → spatial differences can be attributed to geometry alone.